# daStrudel — музыка из notebook

Один проигрыватель, общий таймлайн и замена именованных паттернов на границе цикла.

Перед запуском укажи путь к `daslang-live` через переменную окружения
`DASLANG_LIVE` (до запуска Jupyter), добавь исполняемый файл в `PATH`,
либо допиши `--exe "путь/к/daslang-live.exe"` в первую ячейку `%%strudel`.


## 1. Определить партии

`%%strudel` компилирует функции, но ещё не включает их звучание.
Внешние сэмплы и SoundFont не нужны: здесь используются встроенные синтезаторы.

Состав: выразительный бас, мелодия, мягкие аккорды Am/Am7 и ударные.
Аккорды используют ноты A–C–E–G, совместимые с басом и обоими вариантами мелодии.
Группа `drums` содержит бочку, малый барабан и хай-хэт.
Для отдельной остановки используй `strudel_stop("chords")` или `strudel_stop("drums")`.

In [ ]:
%%strudel
def bass() {
    strudel_add_track(
        note("a2 [~ a2] e2 [g2 a3]", "sawtooth")
            |> lpf(850)
            |> attack(0.008) |> decay(0.18)
            |> sustain(0.45) |> release(0.12)
            |> gain(0.26)
    )
}

def melody() {
    strudel_add_track(
        note("a4 ~ c5 e5", "triangle") |> gain(0.08)
    )
}

def chords() {
    strudel_add_track(
        note("[a3,c4,e4] ~ [a3,c4,e4,g4] ~", "triangle")
            |> attack(0.08) |> release(0.6)
            |> gain(0.035)
    )
}

def drums() {
    strudel_add_track(
        s("bd ~ bd ~") |> gain(0.22)
    )
    strudel_add_track(
        s("~ sd ~ sd") |> gain(0.10)
    )
    strudel_add_track(
        s("hh [hh hh] hh [hh hh]") |> gain(0.045)
    )
}

## 2. Включить музыку

Все четыре партии используют один счётчик времени. Если вызовы попадут по разные стороны
границы цикла, партии вступят в разных циклах, но сохранят общую фазу.

In [ ]:
bass()
melody()
chords()
drums()

## 3. Изменить звучащую мелодию

Выполни следующую ячейку, пока играет музыка. Новая версия `melody` включится
автоматически на границе цикла. Бас продолжит звучать без перезапуска.
Меняй строку нот и повторяй ячейку — повторно вызывать `melody()` не нужно.

In [ ]:
%%strudel
def melody() {
    strudel_add_track(
        note("a4 ~ c5 e5", "triangle) |> gain(0.08)
    )
}

## 4. Остановить только мелодию

Новые ноты прекратятся на границе цикла; уже звучащие ноты и хвосты завершатся
естественно. Определение функции останется доступным, бас продолжит играть.

In [ ]:
strudel_stop("melody")

## 5. Вернуть мелодию

Включается последняя скомпилированная версия, синхронно с общим таймлайном.
Повторный вызов не добавляет дубликат партии.

In [ ]:
melody()

## 6. Посмотреть состояние

`cycle`, `seconds` и `cps` описывают общий транспорт. В `parts` находятся версии
паттернов, ожидающие переключения (`pending`, `waiting`) и границы переключения
(`boundary`).

In [ ]:
strudel_status()

## 7. Завершить

Закрывает единственный проигрыватель этого notebook. Чтобы начать снова,
повтори ячейку определения партий, затем ячейку их включения.

In [ ]:
%%strudel --stop

### Как читать паттерны

- `a2`, `c5`, `e5` — ноты с номером октавы.
- `~` — пауза.
- `[a4 c5]` — две ноты внутри одного шага.
- `gain(0.08)` — громкость.

В `%%strudel` используй функции без аргументов и возвращаемого значения.
Функция строит паттерн при компиляции; обычный вызов `melody()` активирует его.
Переменные из обычных ячеек notebook в музыкальные функции не передаются.